> ### ⚠️ Select the **`Python 3 (croprow)`** kernel first
> This notebook runs in the isolated croprow env (Python **3.11**, OpenCV). If the first cell throws `ModuleNotFoundError: No module named 'cv2'`, the wrong interpreter is selected.
>
> **VSCode:** click **Select Kernel** (top-right) → **Jupyter Kernel** → **`Python 3 (croprow)`**, or **Python Environments... → Enter interpreter path...** and paste `croprow\.venv\Scripts\python.exe`.
>
> Do **not** use the repo-root `.venv` — that is the potato backend (Python 3.13, no cv2 by design). `croprow_disease` shares the `croprow` env; it needs no venv of its own.

# 01b — Use a **provided** healthy/unhealthy dataset

**This is the main path.** Notebook `01_dataset_prep` bootstraps from LettuceMOTS, which contains no affected plants and therefore cannot teach `unhealthy`. When the real two-class dataset arrives, point this notebook at it instead.

It emits the *same* `data/health.yaml` that 01 does, so **notebooks 03–08 work unchanged** and neither knows nor cares which path produced the yaml.

Three layouts are accepted — set `DATASET_ROOT` and run:

| layout | what it looks like | what happens |
| --- | --- | --- |
| `yaml` | a `data.yaml` / `dataset.yaml` at the root | validated and adopted; split taken from the yaml |
| `split` | `images/train` + `images/val` with a mirrored `labels/` tree | adopted as-is (`valid` / `test` also recognised — Roboflow exports use `valid`) |
| `flat` | `images/` + `labels/` with no split | a train/val split is generated, **grouped** so frames from one clip stay on one side |

Images are never copied or modified — the list files point at them where they are.

### What this notebook is strict about

- **Class order.** `0 = healthy`, `1 = unhealthy`. Ultralytics matches classes by *index*, so a dataset labelled `[unhealthy, healthy]` trains perfectly happily and inverts every prediction with no warning. That is checked and rejected.
- **Missing label files.** Ultralytics reads an image with no label as "contains nothing", which actively teaches the model to miss plants. A genuinely empty label file (a background frame with no plants) is fine and is counted separately.
- **An empty class.** Zero `unhealthy` instances means there is nothing to learn and nothing to validate against.

Set `STRICT = False` only after reading the report and deciding the damage is acceptable.

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

from croprow_disease import dataset as D

# ===================== CONFIG (edit here only) =====================
# The provided healthy/unhealthy dataset. Keep it OUTSIDE the repo.
DATASET_ROOT = os.environ.get("HEALTH_DATASET", r"D:\croprow_dataset\crop_health")

# Only used when the dataset has no split of its own (the `flat` layout).
VAL_FRAC = 0.25
SEED = 42

# False = proceed despite defects. Read the report first.
STRICT = True
# ===================================================================
print("dataset root:", DATASET_ROOT)
print("will write  :", DATA_DIR / "health.yaml")

## 1. What are we looking at?

Detects the layout and reports it before touching anything, so a wrong `DATASET_ROOT` fails here with a readable message rather than halfway through.

In [ ]:
root = Path(DATASET_ROOT)
if not root.is_dir():
    raise FileNotFoundError(
        f"Dataset root not found: {root}\n"
        "Set HEALTH_DATASET or edit DATASET_ROOT in the config cell above.")

layout = D.detect_layout(root)
print("layout:", layout)
print("top level:", sorted(p.name for p in root.iterdir())[:15])

src_yaml = D.find_dataset_yaml(root)
if src_yaml:
    print("\nits own yaml:", src_yaml)
    print(src_yaml.read_text())

## 2. Validate + emit `health.yaml`

Scans every label file, checks the class names and order, splits if needed, and writes `data/train.txt`, `data/val.txt` and `data/health.yaml`.

In [ ]:
report = D.prepare_provided_dataset(
    root, DATA_DIR, val_frac=VAL_FRAC, seed=SEED, strict=STRICT)
print(D.format_report(report))

## 3. Class balance — is it trainable?

Both classes need a real population. A model trained where `unhealthy` is a rounding error predicts `healthy` for everything and still posts a respectable mean mAP.

In [ ]:
totals = {
    "healthy": report["train"]["healthy"] + report["val"]["healthy"],
    "unhealthy": report["train"]["unhealthy"] + report["val"]["unhealthy"],
}
verdict = U.check_class_balance(totals)
print(verdict["message"])

print("\nper split:")
for split in ("train", "val"):
    s = report[split]
    n = s["healthy"] + s["unhealthy"]
    print(f"  {split:5s}: {s['images']:5d} images | "
          f"healthy {s['healthy']:6d} ({100 * s['healthy'] / max(n, 1):5.1f}%) | "
          f"unhealthy {s['unhealthy']:6d} ({100 * s['unhealthy'] / max(n, 1):5.1f}%)")

if report["val"]["unhealthy"] < 30:
    print("\nNOTE: fewer than ~30 unhealthy instances in val. mAP on a handful "
          "of positives swings wildly between runs -- treat it as a smoke test, "
          "not a measurement.")

## 4. Eyeball the labels

The cheapest way to catch an inverted class map or a bad export. **Green = healthy, orange-red = unhealthy.** If the colours look wrong on these crops, stop — everything downstream inherits the error.

In [ ]:
%matplotlib inline
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2

images_root = Path(report["images_root"])
labels_root = Path(report["labels_root"])
train_list = [Path(l) for l in Path(report["train_list"]).read_text().splitlines() if l.strip()]

def load_instances(img_path):
    lab = D.label_for_image(img_path, images_root, labels_root)
    out = []
    if lab.is_file():
        for line in lab.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                out.append({"cls": int(float(p[0])), "cx": float(p[1]),
                            "cy": float(p[2]), "w": float(p[3]), "h": float(p[4]),
                            "score": float("nan")})
    return out

# Prefer frames that actually contain an unhealthy instance -- those are the
# ones worth checking, and a random sample of a skewed set rarely shows any.
rng = random.Random(SEED)
pool = rng.sample(train_list, min(400, len(train_list)))
with_unhealthy = [p for p in pool
                  if any(i["cls"] == U.UNHEALTHY for i in load_instances(p))]
print(f"{len(with_unhealthy)} of {len(pool)} sampled train frames contain an "
      f"unhealthy instance")

show = (with_unhealthy[:6] + [p for p in pool if p not in with_unhealthy][:3])[:9]
cols = 3
rows = (len(show) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.8 * rows))
for ax, img_path in zip(np.atleast_1d(axes).ravel(), show):
    inst = load_instances(img_path)
    bgr = U.imread_bgr(img_path)
    drawn = U.draw_instances(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), inst,
                             rgb=True, show_score=False)
    n_u = sum(1 for i in inst if i["cls"] == U.UNHEALTHY)
    ax.imshow(drawn)
    ax.set_title(f"{img_path.name}  |  {len(inst) - n_u} healthy, {n_u} unhealthy",
                 fontsize=8)
    ax.axis("off")
for ax in np.atleast_1d(axes).ravel()[len(show):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Next

`data/health.yaml` now points at the provided dataset. Go straight to **`03_train`** — it reads that yaml and needs no changes.

If you also ran `01_dataset_prep` earlier, note that both write the same `health.yaml`: whichever ran last wins. Re-run this notebook to point back at the provided dataset.